In [1]:
%%script false --no-raise-error
!pip3 install opencv-python numpy matplotlib opendatasets pandas kagglehub ace_tools

In [2]:
%%script false --no-raise-error
!kaggle datasets download -d adamelkholy/human-ai-artwork
!mkdir data
!unzip human-ai-artwork.zip -d data
!rm *.zip

In [3]:
## Package Imports

In [4]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Hide INFO & WARNING logs

import tensorflow as tf
tf.get_logger().setLevel("ERROR")  # Only show errors

import time
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np 

Future exception was never retrieved
future: <Future finished exception=BrokenPipeError(32, 'Broken pipe')>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/unix_events.py", line 676, in write
    n = os.write(self._fileno, data)
BrokenPipeError: [Errno 32] Broken pipe


## Pre-Processing

In [5]:
### Splits and Tuning

In [6]:
weights = {0: 1.0, 1: 1.0} 
num_classes = 1
batch_size = 32
img_ht = 256
img_wt = 256
data_dir = "./data/data"
model_path = "./"

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.4,
    subset="training",
    seed=104,
    image_size=(img_ht, img_wt),
    batch_size = batch_size
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.4,
    subset="validation",
    seed=104,
    image_size=(img_ht, img_wt),
    batch_size=batch_size
)

val_batches = int(0.5 * len(val_test_ds))
val_ds = val_test_ds.take(val_batches)
test_ds = val_test_ds.skip(val_batches)

Found 271993 files belonging to 52 classes.
Using 163196 files for training.


I0000 00:00:1739278060.042320  365362 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739278060.042573  365362 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739278060.070112  365362 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739278060.070317  365362 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

Found 271993 files belonging to 52 classes.
Using 108797 files for validation.


### Imbalance Bias

In [7]:
pos = 190549
neg = 81457
in_bias = np.log([pos/neg])
out_bias = tf.keras.initializers.Constant(in_bias)

### Image to Binary Mappings

In [8]:
def image_to_binary(img, label):
    label = tf.cast(label, tf.int32)
    new_label = tf.where(label < 25, 1, 0) # label is number of ai class folders
    new_label = tf.expand_dims(new_label, axis=-1)
    return (img, new_label)


train_ds = train_ds.map(image_to_binary)
val_ds = val_ds.map(image_to_binary)
test_ds = test_ds.map(image_to_binary)

In [9]:
## Data Augmentation

In [10]:
def img_augmentation(img, lbl):
    min_scale = 0.5
    max_scale = 2.0
    scale_factor = tf.random.uniform(shape=[], minval=min_scale, maxval=max_scale)
    resized_img = tf.image.resize(img, tf.cast(tf.cast(tf.shape(img)[1:3], tf.float32) * scale_factor, tf.int32))
    rescaled_img = tf.image.resize(resized_img, tf.shape(img)[1:3])
    
    return (rescaled_img, lbl)

train_ds = train_ds.map(img_augmentation)
val_ds = val_ds.map(img_augmentation)
test_ds = test_ds.map(img_augmentation)

## Model Training

### Define Models for Experimenting

In [17]:
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, ResNet50, NASNetMobile, DenseNet121, Xception, VGG16
from tensorflow.keras import layers, models

# Function to Create a Binary Classification Model
def create_binary_model(base_model, model_name):
    base_model.trainable = False  # Freeze pre-trained layers
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.Dense(1, activation="sigmoid")(x)  # Binary classification output
    model = models.Model(inputs=base_model.input, outputs=x, name=model_name)
    return model

# Load Models with `include_top=False` to Customize Output
resnet_base = ResNet50(weights="imagenet", include_top=False, input_shape=(256, 256, 3))
resnet_model = create_binary_model(resnet_base, "ResNet50")

densenet_base = DenseNet121(weights="imagenet", include_top=False, input_shape=(256, 256, 3))
densenet_model = create_binary_model(densenet_base, "DenseNet121")

xception_base = Xception(weights="imagenet", include_top=False, input_shape=(256, 256, 3))
xception_model = create_binary_model(xception_base, "Xception")

vgg16_base = VGG16(weights="imagenet", include_top=False, input_shape=(256, 256, 3))
vgg16_model = create_binary_model(vgg16_base, "VGG16")

mobilenet_base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(256, 256, 3))
mobilenet_model = create_binary_model(mobilenet_base, "MobileNetV2")

nasnet_base = NASNetMobile(weights="imagenet", include_top=False, input_shape=(256, 256, 3))
nasnet_model = create_binary_model(nasnet_base, "NASNetMobile")

# Vision Transformer (ViT) - Custom Model with Correct Output
def build_vit_model():
    inputs = layers.Input(shape=(256, 256, 3))
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)  # Fixed for binary classification
    model = models.Model(inputs, outputs, name="VisionTransformer")
    return model

vit_model = build_vit_model()

# Custom CNN (DejAIvu Model) - Already Supports Binary Classification
dejaivu_model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255),

    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation="sigmoid")  # Binary classification
])

dejaivu_model.name = "dejAIvu_model"

# Store models in dictionary
models_dict = {
    "ResNet50": resnet_model,
    "DenseNet121": densenet_model,
    "Xception": xception_model,
    "VGG16": vgg16_model,
    "MobileNetV2": mobilenet_model,
    "NASNetMobile": nasnet_model,
    "VisionTransformer": vit_model,
    "dejAIvu_model": dejaivu_model
}

best_model = resnet_model
best_accuracy = 0.9714
best_model_name = "ResNet50"

results = {}

/tmp/ipykernel_365362/24934264.py:25: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  mobilenet_base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(256, 256, 3))


### Metrics

In [12]:
metrics = [
      tf.keras.metrics.BinaryAccuracy(name='accuracy'),
      tf.keras.metrics.BinaryCrossentropy(name='cross entropy'), # equiv. to model's loss
      tf.keras.metrics.MeanSquaredError(name='MSE'),
      tf.keras.metrics.TruePositives(name='tp'),
      tf.keras.metrics.FalsePositives(name='fp'),
      tf.keras.metrics.TrueNegatives(name='tn'),
      tf.keras.metrics.FalseNegatives(name='fn'),
      tf.keras.metrics.Precision(name='precision'),
      tf.keras.metrics.Recall(name='recall'),
      tf.keras.metrics.AUC(name='roc', curve='ROC'),             # receiver operating characteristic curve
      tf.keras.metrics.AUC(name='prc', curve='PR'),              # precision-recall curve
]

class CustomHistory(tf.keras.callbacks.Callback):
    def __init__(self):
      super(CustomHistory, self).__init__()
      self.losses = []
      self.prcs = []
      self.recalls = []
      self.precisions = []
      self.accuracies = []
      self.mses = []

    """ called upon completion of each batch during training, records all performance metrics """
    def on_train_batch_end(self, batch, logs=None):
      self.losses.append(logs['loss'])
      self.mses.append(logs['MSE'])
      self.accuracies.append(logs['accuracy'])
      self.prcs.append(logs['prc'])
      self.recalls.append(logs['recall'])
      self.precisions.append(logs['precision'])

    """ called upon completion of each batch during testing, records all performance metrics """
    def on_test_batch_end(self, batch, logs=None):
      self.losses.append(logs['loss'])
      self.mses.append(logs['MSE'])
      self.accuracies.append(logs['accuracy'])
      self.prcs.append(logs['prc'])
      self.recalls.append(logs['recall'])
      self.precisions.append(logs['precision'])

    """ return all performance metrics """
    def get_metrics(self):
      return self.losses, self.mses, self.accuracies, self.prcs, self.recalls, self.precisions

## Model Training

### Compiling and Evaluating Helpers

In [13]:
def compile_model(model):
  model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=metrics
  )
  return model

def fit_model(model):
  history_callback = CustomHistory()
  model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    class_weight=weights,
    callbacks=[history_callback]
  )
  return model, history_callback

def evaluate_model_on_test(model):
  eval_metrics = model.evaluate(test_ds)
  return eval_metrics

def save_model(model):
  model_name = model.name
  print("\nSaving " + model_name + ".keras")
  try:
    model.save(model_path + model_name + ".keras")
  except:
    print("Error saving " + model_name + ".keras...")
    return
  print(model_name + ".keras saved successfully.\n")
  
def save_data(data, filename):
  print("Saving data for " + filename)
  try:
    with open(model_path + filename+".txt", 'w') as writefile:
      writefile.write(str(data))
  except:
    print("Error saving data for " + filename)
    return
  print("Data saved.\n")
  
def load_evals(filename):
    file_path = f"{filename}.txt"
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Evaluation file {file_path} not found.")

    with open(file_path, "r") as file:
        evals = eval(file.readline().strip())  # Read and convert string list to actual list

    if not isinstance(evals, list):
        raise ValueError(f"Invalid format in {file_path}. Expected a list of numbers.")

    return {
        "loss": evals[0],
        "accuracy": evals[1],
        "val_loss": evals[2],
        "val_accuracy": evals[3],
        "true_positives": evals[4],
        "false_positives": evals[5],
        "true_negatives": evals[6],
        "false_negatives": evals[7],
        "precision": evals[8],
        "recall": evals[9],
        "f1_score": evals[10],
        "auc": evals[11],
    }

# Load training history from history.txt (formatted as a list of lists)
def load_history(filename):
    file_path = f"{filename}.txt"
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"History file {file_path} not found.")

    with open(file_path, "r") as file:
        history = eval(file.readline().strip())  # Read and convert string list to actual list

    if not isinstance(history, list) or len(history) < 6:
        raise ValueError(f"Invalid format in {file_path}. Expected a list of metric lists.")

    return {
        "losses": history[0],
        "mses": history[1],  # Mean Squared Errors
        "accuracies": history[2],
        "prcs": history[3],  # Precision over time
        "recalls": history[4],
        "precisions": history[5],
    }

### Execute Training


In [14]:
from tensorflow.keras.metrics import MeanSquaredError, AUC, Recall, Precision

def train_model_pipeline(model, mname, lr=0.001):
  start = time.time()
  model_name = model.name
  print("Now training " + model_name)

  # compile model and fit to training data
  compiled_model = compile_model(model)
  compiled_model.optimizer.learning_rate = lr
  trained_model, history = fit_model(compiled_model)

  # save model.keras and history data
  save_data(history.get_metrics(), model_name+"_history")
  save_model(trained_model)

  # evaluate on test set and save evaluation
  evals = evaluate_model_on_test(trained_model)
  save_data(evals, model_name + "_evals")

  time_taken = time.time() - start
  
  final_accuracy = evals[1]
  
  print(f"Training complete for {model_name} in", round((time_taken)/60, 2), "minutes")
  return trained_model, evals, history, final_accuracy


In [15]:
%%script false --no-raise-error
for model_name, model in models_dict.items():
    print(f"Training {model_name}...")
    trained_model, evals, history, final_accuracy = train_model_pipeline(model, model_name)
    results[model_name] = {"evals": evals, "history": history.get_metrics(), "accuracy": final_accuracy}
    
    if final_accuracy > best_accuracy:
        best_accuracy = final_accuracy
        best_model = trained_model
        best_model_name = model_name

if best_model is not None:
    print(f"Best model is {best_model_name} with accuracy: {best_accuracy:.4f}")
    best_model.save(f"best_model_{best_model_name}.keras")

## Display Metrics

### Tables

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display  # Alternative 1

# Prepare results for table
data = []
for model_name, _ in results.items():
    evals = load_evals(f"{model_name}_evals")
    history = load_history(f"{model_name}_history")

    # Extract final epoch metrics
    last_epoch_metrics = {
        "Model": model_name,
        "Accuracy": evals["accuracy"],
        "Loss": evals["loss"],
        "Precision": evals["precision"],
        "Recall": evals["recall"]
    }
    data.append(last_epoch_metrics)

# Convert to DataFrame
df = pd.DataFrame(data)

# Highlight best model
best_model_name = df.loc[df["Accuracy"].idxmax(), "Model"]  # Get the best model by Accuracy
df["Best Model"] = df["Model"] == best_model_name  # Boolean column for best model

display(df)

df_sorted = df.sort_values(by="Accuracy", ascending=False)  # Sort models by accuracy

plt.figure(figsize=(10, 5))
plt.bar(df_sorted["Model"], df_sorted["Accuracy"], color=["blue" if model != best_model_name else "green" for model in df_sorted["Model"]])
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")
plt.xticks(rotation=45)
plt.ylim(0, 1)  # Ensure accuracy stays within [0,1]
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Highlight the best model in green
for i, acc in enumerate(df_sorted["Accuracy"]):
    plt.text(i, acc + 0.01, f"{acc:.3f}", ha="center", fontsize=10)

plt.show()


KeyError: 'accuracy'

### Graphs Per Model

In [ ]:
def get_history(path):
  file = open(path, "r")
  for line in file:
    arr = eval(line)
  file.close()
  return arr

def plot_metric_rolling_average_graph(x, y, title, xlabel, ylabel, cutoff=0):
  plt.plot(x[cutoff:], (np.cumsum(y) / x)[cutoff:])
  plt.xlabel(xlabel)
  plt.ylabel(ylabel)
  plt.title(title)
  plt.ylim(0,1)
  plt.show()

for model_name, results in results.items():
    history_path = f"./{model_name}_history.txt"
    history = get_history(history_path)
    losses, mses, accuracies, prcs, recalls, precisions = history
    x = [x for x in range(len(losses))]
    plot_metric_rolling_average_graph(x, losses, f"Average loss for {model_name}", "Batch No.#", "Loss")
    plot_metric_rolling_average_graph(x, accuracies, f"Average accuracy for {model_name}", "Batch No.#", "Accuracy")
    plot_metric_rolling_average_graph(x, recalls, f"Average recall for {model_name}", "Batch No.#", "Recall")
    plot_metric_rolling_average_graph(x, precisions, f"Average precision for {model_name}", "Batch No.#", "Precision")